# Exploring passive spring assistance with the OpenSim Python API

## Overview

This notebook demonstrates how to use the OpenSim Python API to modify a musculoskeletal model and evaluate the biomechanical effects of a passive assistive spring. A point-to-point spring is added between the centers of mass of the torso and pelvis in the `gait2392` model. The same prescribed crouch motion is then analyzed with the original model and with the spring-assisted model.

The comparison uses two common OpenSim analyses:

- **Inverse dynamics** calculates the generalized forces and moments required to reproduce the prescribed motion. Here, the lumbar extension moment is compared before and after adding the spring.
- **Static optimization** distributes the required generalized forces among the model actuators at each time point. Here, selected muscle-force histories are compared between the two models.

## Learning objectives

By the end of the notebook, you should be able to:

1. Load and modify an OpenSim model using Python.
2. Add and configure a `PointToPointSpring` component.
3. Run inverse dynamics and static optimization programmatically.
4. Read OpenSim `.sto` output files and visualize biomechanical results.
5. Compare joint moments and muscle forces between baseline and assisted models.

## Inputs and outputs

The notebook expects `gait2392_simbody.osim` and `crouch4.mot` in the current Colab working directory. The example does not apply external loads, although the analysis functions accept an optional ExternalLoads XML file. Running the notebook creates a spring-modified model, inverse-dynamics `.sto` files, and separate static-optimization result directories for the baseline and spring-assisted conditions.

> **Note:** OpenSim uses SI units unless stated otherwise. In this notebook, spring stiffness is in N/m, spring rest length is in m, joint moments are in N·m, and muscle forces are in N.

In [ ]:
# Install the OpenSim Python package in a fresh Google Colab runtime.
!pip install opensim

In [ ]:
from pathlib import Path

import opensim as osim


def add_spring(model, stiffness, rest_length):
    """Add a point-to-point spring between the torso and pelvis COMs.

    Parameters
    ----------
    model : str or opensim.Model
        Path to an OpenSim model file, or an already loaded model.
    stiffness : float
        Linear spring stiffness in N/m.
    rest_length : float
        Spring length at which the spring force is zero, in metres.

    Returns
    -------
    str
        Path to the modified model, written as ``model_with_spring.osim``.
    """
    # The Model constructor loads a path or copies an existing Model. Copying
    # prevents this function from modifying a caller-owned model in place.
    model = osim.Model(model)

    # Retrieve the two bodies connected by the spring.
    torso = model.getBodySet().get('torso')
    pelvis = model.getBodySet().get('pelvis')

    # get_mass_center() returns the COM expressed in that body's local frame.
    torso_com = torso.get_mass_center()
    pelvis_com = pelvis.get_mass_center()

    # Construct and name a Force component so it appears clearly in the
    # model's ForceSet and in the serialized .osim file.
    spring = osim.PointToPointSpring()
    spring.setName('torso_pelvis_COM_spring')

    # Connect the spring's body sockets to PhysicalFrame components.
    spring.setBody1(torso)
    spring.setBody2(pelvis)

    # point1 and point2 are expressed in body1 and body2, respectively.
    spring.setPoint1(torso_com)
    spring.setPoint2(pelvis_com)

    # Configure the linear force-length relationship.
    spring.setStiffness(stiffness)
    spring.set_rest_length(rest_length)

    # addForce() transfers the new Force component into the model.
    model.addForce(spring)

    # Resolve all component sockets after changing the model topology.
    model.finalizeConnections()

    # Serialize the modified model so it can also be opened in the OpenSim GUI.
    output_model = 'model_with_spring.osim'
    model.printToXML(output_model)
    return output_model


def inverse_dynamics(
    model,
    kinematic_file,
    external_loads_file=None,
    output_file='id_results.sto',
    cutoff_frequency=4,
    excluded_forces=('Muscles',),
):
    """Compute generalized joint forces with OpenSim inverse dynamics.

    Parameters
    ----------
    model : str or opensim.Model
        Model file path or loaded model used for the analysis.
    kinematic_file : str
        OpenSim .mot file containing coordinate values versus time.
    external_loads_file : str or None, optional
        ExternalLoads XML file, typically referencing measured ground reaction
        forces. If None, inverse dynamics is run without external loads.
    output_file : str, optional
        Destination .sto file for the generalized forces and moments.
    cutoff_frequency : float, optional
        Low-pass cutoff frequency for the coordinates, in Hz.
    excluded_forces : iterable of str, optional
        Force groups excluded from the calculation. Excluding ``Muscles`` makes
        the output represent the net generalized forces required by the motion.

    Returns
    -------
    str
        The path supplied as ``output_file``.

    Raises
    ------
    RuntimeError
        If OpenSim reports that the inverse-dynamics analysis failed.
    """
    # Work with a private Model instance, then initialize its Simbody system.
    model = osim.Model(model)
    model.initSystem()

    # InverseDynamicsTool differentiates the prescribed coordinate motion and
    # solves for the generalized forces needed to produce that motion.
    id_tool = osim.InverseDynamicsTool()
    id_tool.setModel(model)
    id_tool.setCoordinatesFileName(kinematic_file)
    id_tool.setLowpassCutoffFrequency(cutoff_frequency)

    # The XML file describes where and how force-plate data act on the model.
    if external_loads_file is not None:
        id_tool.setExternalLoadsFileName(external_loads_file)

    # OpenSim's Python bindings require an ArrayStr rather than a Python list.
    forces_to_exclude = osim.ArrayStr()
    for force_name in excluded_forces:
        forces_to_exclude.append(force_name)
    id_tool.setExcludedForces(forces_to_exclude)

    # Match the analysis interval exactly to the available kinematic data.
    ik_storage = osim.Storage(kinematic_file)
    start_time = ik_storage.getFirstTime()
    end_time = ik_storage.getLastTime()
    id_tool.setStartTime(start_time)
    id_tool.setEndTime(end_time)

    id_tool.setOutputGenForceFileName(output_file)

    # run() may either throw an OpenSim exception or return False. Handle both.
    try:
        success = id_tool.run()
    except Exception as exc:
        raise RuntimeError("Inverse dynamics failed") from exc
    if not success:
        raise RuntimeError("Inverse dynamics failed")
    return output_file


def run_static_optimization(
    model_file, kinematic_file, external_loads_file, output_directory
):
    """Run OpenSim static optimization using filtered kinematics.

    Static optimization resolves the net joint moments from inverse dynamics
    into individual actuator (typically muscle) forces at each time frame.

    Parameters
    ----------
    model_file : str or path-like
        Path to the OpenSim .osim model used for the analysis.
    kinematic_file : str or path-like
        Path to the inverse-kinematics .mot file containing coordinate values.
    external_loads_file : str, path-like, or None
        Path to an ExternalLoads XML file describing measured loads and the
        bodies to which they are applied. Use None when no external loads
        should be included.
    output_directory : str or path-like
        Directory in which OpenSim writes the static-optimization results.
        The directory is created if it does not already exist.

    Returns
    -------
    str
        The output-directory path.

    Raises
    ------
    RuntimeError
        If OpenSim reports that the AnalyzeTool failed.

    Notes
    -----
    The coordinate trajectories are explicitly low-pass filtered at 2 Hz
    before they are converted into state trajectories. The model must contain
    actuators (and, when needed, reserve actuators) for the optimization to find
    a dynamically feasible solution.
    """
    output_directory = Path(output_directory)
    output_directory.mkdir(parents=True, exist_ok=True)

    # Load a private model instance so adding an Analysis component does not
    # modify any Model object used elsewhere in the notebook.
    model = osim.Model(str(model_file))

    # Use Storage to obtain the exact time interval covered by the IK data.
    kinematics = osim.Storage(str(kinematic_file))
    start_time = kinematics.getFirstTime()
    end_time = kinematics.getLastTime()

    # Filter the coordinate trajectories before differentiating them to obtain
    # speeds. Storage.lowpassIIR() applies the requested cutoff in Hz.
    kinematics.lowpassIIR(2.0)

    # StaticOptimization is an Analysis component. It evaluates the model's
    # actuator forces independently at every time frame.
    static_optimization = osim.StaticOptimization()
    static_optimization.setName('static_optimization')
    static_optimization.setStartTime(start_time)
    static_optimization.setEndTime(end_time)
    static_optimization.setUseModelForceSet(True)

    # Add the analysis before finalizing component connections and constructing
    # the underlying Simbody system.
    model.addAnalysis(static_optimization)
    model.finalizeConnections()
    state = model.initSystem()

    # AnalyzeTool supplies the prescribed motion and external loads to every
    # Analysis attached to the model. AnalyzeTool calls the end time 'FinalTime'.
    analyze_tool = osim.AnalyzeTool()
    analyze_tool.setName('static_optimization')
    analyze_tool.setModel(model)
    analyze_tool.setCoordinatesFileName(str(kinematic_file))
    if external_loads_file is not None:
        analyze_tool.setExternalLoadsFileName(str(external_loads_file))
    analyze_tool.setResultsDir(str(output_directory))
    analyze_tool.setInitialTime(start_time)
    analyze_tool.setFinalTime(end_time)

    # With an in-memory model, AnalyzeTool requires an explicit states history.
    # Convert the filtered coordinates (and derived speeds) into state variables.
    analyze_tool.setStatesFromMotion(
        state, kinematics, kinematics.isInDegrees()
    )

    # Retain the AnalyzeTool cutoff used by the current workflow. Because the
    # filtered states are supplied explicitly above, the 2 Hz Storage filter
    # controls the state trajectories used by this analysis.
    analyze_tool.setLowpassCutoffFrequency(4.0)

    try:
        success = analyze_tool.run()
    except Exception as exc:
        raise RuntimeError('Static optimization failed') from exc
    if not success:
        raise RuntimeError('Static optimization failed')

    return str(output_directory)

In [ ]:
# Input files for the prescribed crouch-motion example.
model = 'gait2392_simbody.osim'
kinematic_file = 'crouch4.mot'

# Baseline inverse dynamics: use the original model without external loads.
inverse_dynamics(model, kinematic_file, None, 'id_results.sto')

# Add the torso-pelvis spring. OpenSim uses N/m and m for these properties.
stiffness = 5000
rest_length = 0.5
model_with_spring = add_spring(model, stiffness, rest_length)

# Repeat the analysis with only the model's spring component changed.
# Keeping the motion, loads, filter, and time range identical makes the two
# result files directly comparable.
inverse_dynamics(model_with_spring, kinematic_file, None, 'id_results_exo.sto')

In [ ]:
# Estimate muscle forces for the original model.
_ = run_static_optimization(
    model, kinematic_file, None, 'baseline_so_results/'
)

# Repeat static optimization for the spring-assisted model.
_ = run_static_optimization(
    model_with_spring, kinematic_file, None, 'spring_so_results/'
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def _read_opensim_storage(file_path):
    """Read the numeric table from an OpenSim Storage file.

    Parameters
    ----------
    file_path : str or path-like
        Path to an OpenSim .sto or .mot text file.

    Returns
    -------
    numpy.ndarray
        Structured array whose field names are the OpenSim column labels.

    Raises
    ------
    ValueError
        If the file does not contain OpenSim's required ``endheader`` marker.
    """
    # Storage files begin with metadata of variable length. Locate the marker
    # that separates this metadata from the column labels and numeric table.
    with open(file_path, encoding='utf-8') as storage_file:
        for line_number, line in enumerate(storage_file):
            if line.strip().lower() == 'endheader':
                break
        else:
            raise ValueError(f"No 'endheader' line found in {file_path}")

    # names=True uses the first line after the header as structured-array names.
    return np.genfromtxt(
        file_path,
        skip_header=line_number + 1,
        names=True,
        encoding='utf-8',
        deletechars='',
    )


def visualize_lumbar_moment_differences(
    without_springs_file, with_springs_file
):
    """Compare lumbar extension moments with and without the spring.

    Parameters
    ----------
    without_springs_file : str or path-like
        Inverse-dynamics .sto file produced by the original model.
    with_springs_file : str or path-like
        Inverse-dynamics .sto file produced after adding the spring.

    Returns
    -------
    fig : matplotlib.figure.Figure
        Figure containing the baseline and spring-assisted moment curves.
    ax : matplotlib.axes.Axes
        Axes containing the two lumbar-moment curves.
    """
    without_springs = _read_opensim_storage(without_springs_file)
    with_springs = _read_opensim_storage(with_springs_file)

    # Fail early if either result lacks the required time or moment data.
    required_columns = {'time', 'lumbar_extension_moment'}

    for file_path, data in (
        (without_springs_file, without_springs),
        (with_springs_file, with_springs),
    ):
        missing = required_columns.difference(data.dtype.names)
        if missing:
            raise ValueError(f"Missing columns in {file_path}: {sorted(missing)}")

    # Use the spring result's samples as the common plotting time grid.
    time = with_springs['time']

    # Interpolate the baseline result onto the spring-result time grid.
    before = np.interp(
        time,
        without_springs['time'],
        without_springs['lumbar_extension_moment'],
    )
    after = with_springs['lumbar_extension_moment']

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(time, before, label='Without spring', linewidth=2)
    ax.plot(time, after, label='With spring', linewidth=2)
    ax.set_title('Lumbar extension moment')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Moment (N·m)')
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    return fig, ax

fig, ax = visualize_lumbar_moment_differences(
    'id_results.sto', 'id_results_exo.sto'
)

In [ ]:
def visualize_muscle_force(
    baseline_muscle_forces_file, assisted_muscle_forces_file, muscle_name
):
    """Compare one muscle's force before and after adding the spring.

    Parameters
    ----------
    baseline_muscle_forces_file : str or path-like
        Static Optimization force .sto file for the original model.
    assisted_muscle_forces_file : str or path-like
        Static Optimization force .sto file for the spring-assisted model.
    muscle_name : str
        Name of the muscle to plot, for example ``'soleus_r'``. Both plain
        column labels and full OpenSim component paths such as
        ``'/forceset/soleus_r'`` are supported.

    Returns
    -------
    fig : matplotlib.figure.Figure
        Figure containing the baseline and assisted muscle-force curves.
    ax : matplotlib.axes.Axes
        Axes object that can be used to customize the plot further.

    Raises
    ------
    ValueError
        If an input lacks a time column, the requested muscle is absent, or
        more than one column matches the supplied muscle name.

    Notes
    -----
    The baseline force is interpolated onto the assisted result's time grid,
    allowing the two files to use slightly different sample times.
    """
    baseline_muscle_forces = _read_opensim_storage(
        baseline_muscle_forces_file
    )
    assisted_muscle_forces = _read_opensim_storage(
        assisted_muscle_forces_file
    )
    for file_path, data in (
        (baseline_muscle_forces_file, baseline_muscle_forces),
        (assisted_muscle_forces_file, assisted_muscle_forces),
    ):
        if data.dtype.names is None or 'time' not in data.dtype.names:
            raise ValueError(f"No 'time' column found in {file_path}")

    if not isinstance(muscle_name, str) or not muscle_name.strip():
        raise ValueError('muscle_name must be a non-empty string')

    muscle_name = muscle_name.strip()

    # Static Optimization files may use either a simple ForceSet name or a
    # full component path. Compare the requested name with both forms.
    baseline_column_names = baseline_muscle_forces.dtype.names
    matching_columns = [
        column_name
        for column_name in baseline_column_names
        if column_name != 'time'
        and (
            column_name == muscle_name
            or column_name.rsplit('/', 1)[-1] == muscle_name
        )
    ]

    if not matching_columns:
        available_muscles = [
            name.rsplit('/', 1)[-1]
            for name in baseline_column_names
            if name != 'time'
        ]
        preview = ', '.join(available_muscles[:10])
        if len(available_muscles) > 10:
            preview += ', ...'
        raise ValueError(
            f"Muscle '{muscle_name}' was not found. Available muscles: {preview}"
        )
    if len(matching_columns) > 1:
        raise ValueError(
            f"Muscle name '{muscle_name}' is ambiguous: {matching_columns}"
        )

    baseline_muscle_column = matching_columns[0]

    assisted_column_names = assisted_muscle_forces.dtype.names
    assisted_matches = [
        column_name
        for column_name in assisted_column_names
        if column_name != 'time'
        and (
            column_name == muscle_name
            or column_name.rsplit('/', 1)[-1] == muscle_name
        )
    ]
    if len(assisted_matches) != 1:
        raise ValueError(
            f"Expected one assisted-model column for '{muscle_name}', "
            f"but found {assisted_matches}"
        )
    assisted_muscle_column = assisted_matches[0]

    time = np.atleast_1d(assisted_muscle_forces['time'])
    baseline_force = np.interp(
        time,
        np.atleast_1d(baseline_muscle_forces['time']),
        np.atleast_1d(baseline_muscle_forces[baseline_muscle_column]),
    )
    assisted_force = np.atleast_1d(
        assisted_muscle_forces[assisted_muscle_column]
    )

    # Muscle force is reported in newtons by OpenSim's SI-unit convention.
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(
        time, baseline_force, color='tab:blue', linewidth=2, label='Baseline'
    )
    ax.plot(
        time, assisted_force, color='tab:red', linewidth=2, label='With spring'
    )
    ax.set_title(f"Static Optimization muscle force: {muscle_name}")
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Muscle force (N)')
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()

    return fig, ax

# Compare the left erector spinae force from both analyses.
baseline_force_file = (
    'baseline_so_results/'
    'static_optimization_static_optimization_force.sto'
)
assisted_force_file = (
    'spring_so_results/'
    'static_optimization_static_optimization_force.sto'
)

fig, ax = visualize_muscle_force(
    baseline_force_file, assisted_force_file, 'ercspn_l'
)